# CIS 536/736: Computer Graphics / Game Engines
## Lab 3a: Geometry Nodes Procedural Asset

**Module 5: Geometry Nodes Fundamentals & Patterns**  
**Target Environment:** Blender 4.2 LTS / Python 3.10+ (pxr)  
**Execution Time:** 120 Minutes

---
### 1. Lab Objective
Transitioning from real-time GPU shading (Unity) to offline procedural geometry generation (Blender). You will construct a non-destructive, parameter-driven asset (e.g., ruined pillar, asteroid, foliage cluster) using the Geometry Nodes modifier. This asset must be explicitly prepared for OpenUSD export to act as a dynamic prop in your Unity HDRP pipeline.

### 2. Base Asset Foundation
1. Open Blender 4.2 LTS.
2. Delete the default cube and add a base primitive (e.g., Plane, Cylinder, or Icosphere) to act as your canvas.
3. Open the **Geometry Nodes** workspace and click **New** to attach a modifier tree to your primitive.
4. Construct the base shape using structural nodes (`Transform`, `Extrude Mesh`, `Subdivision Surface`).

### 3. Point Distribution & Instancing
You must implement an instancing system to drastically increase geometric detail without inflating memory limits.

*   **Scatter:** Add a `Distribute Points on Faces` node. Choose **Poisson Disk** to avoid intersecting instances.
*   **Instance:** Add an `Instance on Points` node.
*   **Variation:** Use `Random Value` nodes mapped to the **Scale** and **Rotation** inputs of your instancer to break visual uniformity.

### 4. Exposing the Parameters (The Art-Directable API)
Your procedural asset must be modular. Hardcoding values violates the structural intent of the lab.

Route the following values from your internal nodes to the blank sockets of the `Group Input` node. They will appear in the Modifier Properties panel:
1.  **Seed:** (Integer) To cycle through random variations.
2.  **Density:** (Float) To control the maximum density of your Poisson Disk distribution.
3.  **Scale Variation:** (Vector/Float) To define the min/max limits of your instanced objects.

### 5. Finalizing the Mesh & Material Assignment
Before OpenUSD can serialize this asset, the instances and materials must be baked into the final output geometry.

*   **Realize Instances:** Add a `Realize Instances` node at the end of your instancing chain. OpenUSD cannot universally parse un-realized nested Blender instances.
*   **Set Material:** Add a `Set Material` node just before the `Group Output`. Assign a base principled BSDF material. (The actual shader will be applied in Unity in Lab 3b).

### 6. Pipeline Validation: Python USD Stage Check
Before you commit, verify that your exported `.usd` or `.usda` file actually contains valid geometry, not just empty transforms resulting from failed Geometry Node evaluations. 

Run the following `pxr` snippet against your exported file to validate the stage topology.

In [ ]:
from pxr import Usd, UsdGeom
import os

# Define the path to your exported procedural asset
usd_file_path = "exports/procedural_asset_v1.usda"

def validate_procedural_usd(file_path):
    if not os.path.exists(file_path):
        print(f"[ERROR] File not found: {file_path}")
        return False
        
    stage = Usd.Stage.Open(file_path)
    mesh_count = 0
    
    print(f"\n--- Validating USD Stage: {file_path} ---")
    
    for prim in stage.Traverse():
        if prim.IsA(UsdGeom.Mesh):
            mesh_count += 1
            # Extract vertex count to ensure Realize Instances worked
            mesh = UsdGeom.Mesh(prim)
            points = mesh.GetPointsAttr().Get()
            point_count = len(points) if points else 0
            print(f"[PASS] Found Mesh Prim: {prim.GetPath()} | Vertices: {point_count}")
            
    if mesh_count == 0:
        print("[FAIL] No UsdGeom.Mesh primitives found. Did you forget 'Realize Instances' in Geometry Nodes?")
        return False
        
    print("\n[SUCCESS] Asset is topologically valid and ready for Unity 6 HDRP import.")
    return True

# Execute Validation
# validate_procedural_usd(usd_file_path)

---
### 7. Submission Checklist & GitOps Protocol
- [ ] Geometry Nodes graph utilizes fields and parameters (no destructive modifiers).
- [ ] Seed, Density, and Scale are exposed to the Modifier Panel.
- [ ] `Realize Instances` is implemented prior to output.
- [ ] Asset exports cleanly to `.usd` format without errors.
- [ ] The `.blend` source and `.usd` export are committed to your `cis536_736_hw` repository on GitHub.
- [ ] The absolute SSOT URL to your commit is posted in the Canvas assignment drop.